# Leveraging Pre-trained Models with Transfer Learning

## Bonus: Changing and tuning only the last layer

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

torch.manual_seed(123)

- Loading a model from the PyTorch Hub: https://pytorch.org/docs/stable/hub.html

In [2]:
entrypoints = torch.hub.list('pytorch/vision:v0.13.0', force_reload=True)
for e in entrypoints:
    if "resnet" in e:
        print(e)

Downloading: "https://github.com/pytorch/vision/zipball/v0.13.0" to /home/zeus/.cache/torch/hub/v0.13.0.zip
deeplabv3_resnet101
deeplabv3_resnet50
fcn_resnet101
fcn_resnet50
resnet101
resnet152
resnet18
resnet34
resnet50
wide_resnet101_2
wide_resnet50_2


In [3]:
pytorch_model = torch.hub.load('pytorch/vision:v0.13.0', 'resnet18', weights='IMAGENET1K_V1')
# Freeze all layers except the final FC
for param in pytorch_model.parameters():
    param.requires_grad = False

pytorch_model.fc = nn.Linear(512, 10)  # new layer has requires_grad=True by default
print(pytorch_model)

Using cache found in /home/zeus/.cache/torch/hub/pytorch_vision_v0.13.0


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/zeus/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 147MB/s] 

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
# Alternatively:

#from torchvision.models import resnet18, ResNet18_Weights

#weights = ResNet18_Weights.IMAGENET1K_V1

## Custom Transform

- Also, we now have to keep in mind the preprocessing protocol that was used for pre-training the model:

In [5]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess_transform = weights.transforms()
print(preprocess_transform)

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [6]:
train_full   = torchvision.datasets.CIFAR10(root="data/", train=True,  download=True,  transform=preprocess_transform)
test_dataset = torchvision.datasets.CIFAR10(root="data/", train=False, download=True,  transform=preprocess_transform)

indices       = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(123))
train_dataset = Subset(train_full, indices[:45000])
val_dataset   = Subset(train_full, indices[45000:])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

  0%|          | 0.00/170M [00:00<?, ?B/s]

100%|██████████| 170M/170M [02:36<00:00, 1.09MB/s] 


In [7]:
def train_one_epoch(model, loader, optimizer, criterion, acc_metric, device):
    model.train()
    acc_metric.reset()
    total_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()


def evaluate(model, loader, criterion, acc_metric, device):
    model.eval()
    acc_metric.reset()
    total_loss = 0.0

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()


In [11]:
torch.manual_seed(123)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = pytorch_model.to(device)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

writer = SummaryWriter(log_dir="logs/resnet18-two-stage")

In [12]:
STAGE1_EPOCHS = 5

print("--- Stage 1: FC only ---")
for epoch in range(1, STAGE1_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, train_acc, device)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, val_acc, device)

    writer.add_scalars("Loss",     {"train": tr_loss, "val": vl_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": tr_acc,  "val": vl_acc},  epoch)
    writer.add_scalar("Stage", 1, epoch)

    print(f"Epoch {epoch:02d}/{STAGE1_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

Epoch 01/5 | Train Loss: 1.6286  Acc: 0.7758 | Val Loss: 1.7004  Acc: 0.7618
Epoch 02/5 | Train Loss: 1.4161  Acc: 0.7845 | Val Loss: 1.5595  Acc: 0.7678
Epoch 03/5 | Train Loss: 1.2824  Acc: 0.7875 | Val Loss: 1.4847  Acc: 0.7644
Epoch 04/5 | Train Loss: 1.1924  Acc: 0.7882 | Val Loss: 1.4252  Acc: 0.7660
Epoch 05/5 | Train Loss: 1.1173  Acc: 0.7880 | Val Loss: 1.3556  Acc: 0.7666


In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

STAGE2_EPOCHS = 45
TOTAL_EPOCHS  = STAGE1_EPOCHS + STAGE2_EPOCHS

In [ ]:
print("\n--- Stage 2: Full fine-tuning ---")
for epoch in range(STAGE1_EPOCHS + 1, TOTAL_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, train_acc, device)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, val_acc, device)

    writer.add_scalars("Loss",     {"train": tr_loss, "val": vl_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": tr_acc,  "val": vl_acc},  epoch)
    writer.add_scalar("Stage", 2, epoch)

    print(f"Epoch {epoch:02d}/{TOTAL_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

writer.close()

In [ ]:
test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
te_loss, te_acc = evaluate(model, test_loader, criterion, test_acc, device)
print(f"\nTest Loss: {te_loss:.4f} | Test Accuracy: {te_acc:.4f}")